In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!wget -O /kaggle/working/ntl_np_day.rar https://huggingface.co/datasets/myyyyw/NTLNP/resolve/main/voc_day.rar

In [ ]:
!unrar x /kaggle/working/ntl_np_day.rar /kaggle/working/NTLNP_Day/

In [ ]:
import os

ann_path = "/kaggle/working/NTLNP_Day/voc_day/Annotations"
img_path = "/kaggle/working/NTLNP_Day/voc_day/JPEGImages"

print("XML annotations:", len(os.listdir(ann_path)))
print("Images:", len(os.listdir(img_path)))

In [ ]:
import os
import xml.etree.ElementTree as ET
from collections import Counter

ann_path = "/kaggle/working/NTLNP_Day/voc_day/Annotations"

classes = Counter()

for file in os.listdir(ann_path):
    if file.endswith(".xml"):
        root = ET.parse(os.path.join(ann_path, file)).getroot()

        for obj in root.findall("object"):
            name = obj.find("name").text
            classes[name] += 1

print("Number of classes:", len(classes))
print("\nClasses:")
for name, count in sorted(classes.items()):
    print(name, ":", count)

In [ ]:
!wget -O /kaggle/working/ntl_np_night.rar https://huggingface.co/datasets/myyyyw/NTLNP/resolve/main/voc_night.rar

In [ ]:
!unrar x /kaggle/working/ntl_np_night.rar /kaggle/working/NTLNP_Night/

In [ ]:
import os

ann_path = "/kaggle/working/NTLNP_Night/voc_night/Annotations"
img_path = "/kaggle/working/NTLNP_Night/voc_night/JPEGImages"

print("XML annotations:", len(os.listdir(ann_path)))
print("Images:", len(os.listdir(img_path)))

In [ ]:
import os
import xml.etree.ElementTree as ET
from collections import Counter

ann_path = "/kaggle/working/NTLNP_Night/voc_night/Annotations"

classes = Counter()

for file in os.listdir(ann_path):
    if file.endswith(".xml"):
        root = ET.parse(os.path.join(ann_path, file)).getroot()

        for obj in root.findall("object"):
            name = obj.find("name").text
            classes[name] += 1

print("Number of classes:", len(classes))
print("\nClasses:")
for name, count in sorted(classes.items()):
    print(name, ":", count)

In [ ]:
import os
import xml.etree.ElementTree as ET

ann_path = "/kaggle/working/NTLNP_Day/voc_day/Annotations"

# Pick one XML file
sample = sorted(
    [f for f in os.listdir(ann_path) if f.endswith(".xml")]
)[0]

# Read XML
xml_file = os.path.join(ann_path, sample)
root = ET.parse(xml_file).getroot()

print("Sample file:", sample)

# Image dimensions
width = root.find("size/width").text
height = root.find("size/height").text

print("Image size:", width, "x", height)

# Objects
print("\nObjects:")

for obj in root.findall("object"):
    name = obj.find("name").text

    box = obj.find("bndbox")
    xmin = box.find("xmin").text
    ymin = box.find("ymin").text
    xmax = box.find("xmax").text
    ymax = box.find("ymax").text

    print("Class:", name)
    print("Bounding box:", xmin, ymin, xmax, ymax)
    print()

In [ ]:
import os

base = "/kaggle/working/NTLNP_YOLO"

folders = [
    "images/train",
    "images/val",
    "images/test",
    "labels/train",
    "labels/val",
    "labels/test"
]

for folder in folders:
    os.makedirs(os.path.join(base, folder), exist_ok=True)

print("YOLO dataset folders created successfully!")

In [ ]:
classes = [
    "AmurTiger",
    "Badger",
    "BlackBear",
    "Cow",
    "Dog",
    "Hare",
    "Leopard",
    "LeopardCat",
    "MuskDeer",
    "RaccoonDog",
    "RedFox",
    "RoeDeer",
    "Sable",
    "SikaDeer",
    "Weasel",
    "WildBoar",
    "Y.T.Marten"
]

class_to_id = {name: i for i, name in enumerate(classes)}

print("Number of classes:", len(classes))
print(class_to_id)

In [ ]:
import os
import random
import shutil
import xml.etree.ElementTree as ET

# -----------------------------
# Paths
# -----------------------------
day_ann = "/kaggle/working/NTLNP_Day/voc_day/Annotations"
day_img = "/kaggle/working/NTLNP_Day/voc_day/JPEGImages"

night_ann = "/kaggle/working/NTLNP_Night/voc_night/Annotations"
night_img = "/kaggle/working/NTLNP_Night/voc_night/JPEGImages"

base = "/kaggle/working/NTLNP_YOLO"

# -----------------------------
# Class mapping
# -----------------------------
classes = [
    "AmurTiger",
    "Badger",
    "BlackBear",
    "Cow",
    "Dog",
    "Hare",
    "Leopard",
    "LeopardCat",
    "MuskDeer",
    "RaccoonDog",
    "RedFox",
    "RoeDeer",
    "Sable",
    "SikaDeer",
    "Weasel",
    "WildBoar",
    "Y.T.Marten"
]

class_to_id = {name: i for i, name in enumerate(classes)}

# -----------------------------
# Collect Day + Night files
# Prefix prevents duplicate filenames
# -----------------------------
samples = []

for ann_file in os.listdir(day_ann):
    if ann_file.endswith(".xml"):
        samples.append((
            "day",
            os.path.join(day_ann, ann_file),
            day_img,
            "day_" + os.path.splitext(ann_file)[0]
        ))

for ann_file in os.listdir(night_ann):
    if ann_file.endswith(".xml"):
        samples.append((
            "night",
            os.path.join(night_ann, ann_file),
            night_img,
            "night_" + os.path.splitext(ann_file)[0]
        ))

print("Total samples:", len(samples))

# -----------------------------
# Shuffle and split
# 70% train, 20% validation, 10% test
# -----------------------------
random.seed(42)
random.shuffle(samples)

n = len(samples)

train_end = int(0.70 * n)
val_end = int(0.90 * n)

splits = {
    "train": samples[:train_end],
    "val": samples[train_end:val_end],
    "test": samples[val_end:]
}

print("Train:", len(splits["train"]))
print("Validation:", len(splits["val"]))
print("Test:", len(splits["test"]))

# -----------------------------
# Convert XML → YOLO
# -----------------------------
total_objects = 0
bad_objects = 0

for split, split_samples in splits.items():

    image_out = os.path.join(base, "images", split)
    label_out = os.path.join(base, "labels", split)

    for source_type, xml_path, source_img_dir, new_name in split_samples:

        root = ET.parse(xml_path).getroot()

        # Image dimensions
        width = float(root.find("size/width").text)
        height = float(root.find("size/height").text)

        # Find original image
        original_base = os.path.splitext(
            os.path.basename(xml_path)
        )[0]

        image_path = None

        for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
            candidate = os.path.join(
                source_img_dir,
                original_base + ext
            )

            if os.path.exists(candidate):
                image_path = candidate
                break

        if image_path is None:
            print("Image not found:", original_base)
            continue

        # Create symbolic link to original image
        image_ext = os.path.splitext(image_path)[1]
        output_image = os.path.join(
            image_out,
            new_name + image_ext
        )

        if not os.path.exists(output_image):
            os.symlink(image_path, output_image)

        # YOLO label file
        label_path = os.path.join(
            label_out,
            new_name + ".txt"
        )

        with open(label_path, "w") as f:

            for obj in root.findall("object"):

                class_name = obj.find("name").text.strip()

                if class_name not in class_to_id:
                    print("Unknown class:", class_name)
                    bad_objects += 1
                    continue

                class_id = class_to_id[class_name]

                box = obj.find("bndbox")

                xmin = float(box.find("xmin").text)
                ymin = float(box.find("ymin").text)
                xmax = float(box.find("xmax").text)
                ymax = float(box.find("ymax").text)

                # Convert Pascal VOC → YOLO
                x_center = ((xmin + xmax) / 2) / width
                y_center = ((ymin + ymax) / 2) / height

                box_width = (xmax - xmin) / width
                box_height = (ymax - ymin) / height

                f.write(
                    f"{class_id} "
                    f"{x_center:.6f} "
                    f"{y_center:.6f} "
                    f"{box_width:.6f} "
                    f"{box_height:.6f}\n"
                )

                total_objects += 1

print("\n==============================")
print("VOC → YOLO conversion complete!")
print("==============================")
print("Total images:", len(samples))
print("Total objects converted:", total_objects)
print("Unknown classes:", bad_objects)

In [ ]:
import os

print("Contents of /kaggle/working:")
print(os.listdir("/kaggle/working"))

In [ ]:
import yaml

data = {
    "path": "/kaggle/working/NTLNP_YOLO",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 17,
    "names": classes
}

yaml_path = "/kaggle/working/NTLNP_YOLO/data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print("data.yaml created successfully!")
print("\nContents:\n")

with open(yaml_path, "r") as f:
    print(f.read())

In [ ]:
import os

base = "/kaggle/working/NTLNP_YOLO"

for split in ["train", "val", "test"]:
    image_dir = os.path.join(base, "images", split)
    label_dir = os.path.join(base, "labels", split)

    images = [
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    labels = [
        f for f in os.listdir(label_dir)
        if f.endswith(".txt")
    ]

    print(f"{split.upper()}")
    print("Images :", len(images))
    print("Labels :", len(labels))
    print()

In [ ]:
import os

label_dir = "/kaggle/working/NTLNP_YOLO/labels/train"

label_files = [
    f for f in os.listdir(label_dir)
    if f.endswith(".txt")
]

sample_label = sorted(label_files)[0]
label_path = os.path.join(label_dir, sample_label)

print("Sample label:", sample_label)
print("\nLabel contents:\n")

with open(label_path, "r") as f:
    print(f.read())

In [ ]:
!pip install -U ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.yaml")

print("YOLO26n architecture created successfully!")

In [ ]:
model.info()

In [ ]:
import os

data_yaml = "/kaggle/working/NTLNP_YOLO/data.yaml"

print("data.yaml exists:", os.path.exists(data_yaml))
print("Path:", data_yaml)

In [ ]:
imgsz = 640

print("Image size:", imgsz)

In [ ]:
import torch

print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
batch = -1

print("Batch size setting:", batch)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.yaml")

results = model.train(
    data="/kaggle/working/NTLNP_YOLO/data.yaml",
    epochs=25,
    imgsz=640,
    batch=16,
    device="0,1",
    project="/kaggle/working/runs",
    name="YOLO26n",
    exist_ok=True
)

In [ ]:
from ultralytics import YOLO

model_s = YOLO("yolo26s.yaml")

print("YOLO26s architecture created successfully!")

In [ ]:
model_s.info()

In [ ]:
from ultralytics import YOLO

model_s = YOLO("yolo26s.yaml")

results_s = model_s.train(
    data="/kaggle/working/NTLNP_YOLO/data.yaml",
    epochs=25,
    imgsz=640,
    batch=16,
    device="0,1",
    project="/kaggle/working/runs",
    name="YOLO26s",
    exist_ok=True
)

In [ ]:
import os

base = "/kaggle/working/runs"

for model in ["YOLO26n", "YOLO26s"]:
    folder = os.path.join(base, model)
    print(f"\n{model}")
    print("best.pt:", os.path.exists(os.path.join(folder, "weights", "best.pt")))
    print("results.csv:", os.path.exists(os.path.join(folder, "results.csv")))

In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 4.4 MB/s eta 0:00:00


In [3]:
import ultralytics

print("Ultralytics version:", ultralytics.__version__)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics version: 8.4.174


In [4]:
from ultralytics import YOLO

# Load YOLO26n
model_n = YOLO("/kaggle/working/yolo26n_best.pt")

# Load YOLO26s
model_s = YOLO("/kaggle/working/yolo26s_best.pt")

print("YOLO26n loaded successfully!")
print("YOLO26s loaded successfully!")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/yolo26n_best.pt'